# Método de Newton aplicado a Brown Almost Linear (ALMOST)

## Gradiente de ALMOST

Ver el archivo `01_max_descenso_ALMOST.ipynb` para el análisis detallado.

## Hessiana de ALMOST

Aplicando la regla de la cadena y teniendo en cuenta que $\partial r_i/\partial x_k=\delta_{ik}+1$ y $\partial^2 r_n/\partial x_k\partial x_l=0$ si $k=l$, $=P_{kl}:=\prod_{j\notin\{k,l\}}x_j$ si $k\neq l$, se obtiene:
$$H_{kl}=2\sum_{i=1}^{n-1}(\delta_{ik}+1)(\delta_{il}+1)+2P_kP_l+2r_n\cdot\begin{cases}P_{kl}, & k\neq l\\ 0, & k=l\end{cases}$$
Evaluando la suma:
- Si $k,l\le n-1$: $H_{kl}=2(n+1+\delta_{kl})+2P_kP_l+2r_nP_{kl}\cdot\mathbb{1}_{k\neq l}$.
- Si $k\le n-1$, $l=n$: $H_{kl}=2n+2P_kP_n+2r_nP_{kl}$.
- $H_{nn}=2(n-1)+2P_n^2$.

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

$S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$ para todo $k$.
$$f(x_0)=3\cdot(-2.5)^2+(-0.9375)^2\approx 19.6289,$$
$$\nabla f(x_0)=(-20.2344,\,-20.2344,\,-20.2344,\,-15.2344)^t,\quad\|\nabla f(x_0)\|\approx 38.2153,$$
$$\nabla^2 f(x_0)=\begin{pmatrix}12.031&9.5625&9.5625&7.5625\\9.5625&12.031&9.5625&7.5625\\9.5625&9.5625&12.031&7.5625\\7.5625&7.5625&7.5625&6.031\end{pmatrix},$$
que es definida positiva. Se resuelve $\nabla^2 f(x_0)d_0=-\nabla f(x_0)$, se actualiza $x_1=x_0+d_0$ y se observa una reducción drástica de \(f\) (de \(19.63\) a valores del orden de \(10^{-2}\)). La convergencia es cuadrática en una vecindad de \(x^*=(1,1,1,1)\), donde \(\nabla^2 f(x^*)\) es definida positiva (autovalores \(2,\,2,\,0.0871,\,45.913\) para \(n=4\)).

In [1]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import almost_f, almost_grad, almost_hess, newton_method

## Método de Newton aplicado a ALMOST

Para los detalles paso a paso del funcionamiento del método de Newton vea el notebook `02_newton_SINGX.ipynb`. 

### Punto de partida y verificación de hipótesis

Se parte de $x_0=(0.5,0.5,0.5,0.5)^t$ ($n=4$). En este punto:
- $f(x_0)\approx 19.6289$ y $\|\nabla f(x_0)\|\approx 38.2153$.
- La Hessiana $\nabla^2 f(x_0)$ es **definida positiva** (todos sus autovalores son positivos), por lo que el sistema de Newton tiene solución única y la dirección $d_0$ es de descenso.

### Resultado de la ejecución

El método converge en **pocas iteraciones** (típicamente menos de 10 para $n=4$) al mínimo global
$$x^*=(1,1,1,1)^t,\qquad f(x^*)=0,$$
partiendo de $x_0=(0.5,0.5,0.5,0.5)^t$. La convergencia es **cuadrática** en una vecindad de $x^*$, consistente con el Teorema 9.3.1: en el óptimo, $\nabla^2 f(x^*)$ es definida positiva (para $n=4$, autovalores $\approx 0.0871,\,2,\,2,\,45.913$, número de condición $\approx 527$).

La primera iteración ya reduce $f$ de $19.63$ a un valor del orden de $10^{-2}$; las siguientes confirman la convergencia cuadrática hasta alcanzar precisión de máquina. **El método no falla** para este punto inicial: la Hessiana se mantiene definida positiva a lo largo de toda la trayectoria y el sistema lineal de Newton está bien condicionado.

### Advertencia sobre la elección del punto inicial

ALMOST posee **múltiples mínimos globales** para $n\ge 2$ (por ejemplo, además de $(1,\dots,1)$, los puntos con $x_1=\dots=x_{n-1}=\alpha$ y $x_n=n+1-n\alpha$ donde $\alpha^{n-1}(n+1-n\alpha)=1$). Además, su Hessiana puede volverse **indefinida** lejos de $x^*$ (por ejemplo, en $(1,1,-1,0.5)$ presenta un autovalor negativo $\approx -3.183$). Si se parte de un punto inicial en una región donde $\nabla^2 f(x_k)$ es indefinida o singular, el método de Newton puede generar una dirección que no sea de descenso o puede fallar al resolver el sistema lineal. En tal caso sería necesario recurrir a una versión globalizada (búsqueda lineal, región de confianza o regularización $[\nabla^2 f(x_k)+\varepsilon_k I]$). Con el punto inicial $x_0=(0.5,0.5,0.5,0.5)^t$ esto **no ocurre** y el método converge sin dificultades.

In [2]:
# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375,)*3 + (-15.234375,)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153
print(f"H(x0)       =\n{np.round(almost_hess(x0), 6)}")
print(f"autovalores = {np.linalg.eigvalsh(almost_hess(x0))}")  # todos > 0
print()

# Newton desde x0
x_opt, hist = newton_method(almost_f, almost_grad, almost_hess, x0,
                            tol=1e-12, max_iter=50, verbose=True)

print(f"\n--- Resultado final ---")
print(f"x_opt      = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)   = {almost_f(x_opt):.2e}")  # esperado: → 0
print(f"||grad||   = {np.linalg.norm(almost_grad(x_opt)):.2e}")
print(f"Iteraciones: {len(hist['f'])-1}")

# Hessiana en el óptimo (debe ser definida positiva)
print(f"\nAutovalores H(1,1,1,1) = {np.linalg.eigvalsh(almost_hess(np.ones(4)))}")
# esperado: ≈ (0.0871, 2, 2, 45.913)  -> todos > 0

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866
H(x0)       =
[[12.03125  9.5625   9.5625   7.5625 ]
 [ 9.5625  12.03125  9.5625   7.5625 ]
 [ 9.5625   9.5625  12.03125  7.5625 ]
 [ 7.5625   7.5625   7.5625   6.03125]]
autovalores = [ 0.44462812  2.46875     2.46875    36.74287188]

k=0: f=6.276738e-02, ||g||=2.457e+00, ||d||=1.197e+00
k=1: f=3.337418e-04, ||g||=5.393e-02, ||d||=5.653e-02
k=2: f=8.077751e+00, ||g||=2.650e+01, ||d||=6.776e+00
k=3: f=3.273639e+00, ||g||=1.006e+01, ||d||=7.180e-01
k=4: f=1.692277e+00, ||g||=3.605e+00, ||d||=5.691e-01
k=5: f=1.174033e+00, ||g||=1.235e+00, ||d||=5.250e-01
k=6: f=1.032566e+00, ||g||=3.763e-01, ||d||=4.317e-01
k=7: f=1.004897e+00, ||g||=1.046e-01, ||d||=2.860e-01
k=8: f=1.000669e+00, ||g||=2.791e-02, ||d||=1.618e-01
k=9: f=1.000088e+00, ||g||=7.256e-03, ||d||=8.578e-02
k=10: f=1.000011e+00, ||g||=1.854e-03, ||d||=4.420e-02
k=11: f=1.000001e+00, ||g||=4.690e-04, ||d||=2.245e-02
k=12